In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("..") / "src"))

import time
import pandas as pd
from prospect.geology import get_geology

DATA = Path("..") / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

In [2]:
df = pd.read_csv(RAW / "mrds-GA.csv", low_memory=False, encoding="latin-1")
print(f"{len(df)} records")

# NOTE: commod1 holds comma-separated lists ("Thorium, REE, Zirconium");
# commod2/3 are nearly empty. Mineral names also hide in ore/gangue —
# that's where Georgia's garnet went (D9).
up = lambda c: df[c].fillna("").astype(str).str.upper()

# The forensics that drove D9, preserved as evidence:
for col in ["commod1", "commod2", "commod3", "ore", "gangue"]:
    print(f"{col:8s}  GARNET: {up(col).str.contains('GARNET').sum():3d}   "
          f"GOLD: {up(col).str.contains('GOLD').sum():3d}")

2657 records
commod1   GARNET:   1   GOLD: 479
commod2   GARNET:   0   GOLD:   3
commod3   GARNET:   0   GOLD:  27
ore       GARNET:   0   GOLD: 349
gangue    GARNET:  10   GOLD:   0


In [3]:
# D9: v0 target = GOLD. Positive = "GOLD" anywhere in commod1..3.
# (ore-field GOLD mentions overlap commod1 almost entirely — not separately included; see D9.)
is_gold = (
    up("commod1").str.contains("GOLD")
    | up("commod2").str.contains("GOLD")
    | up("commod3").str.contains("GOLD")
)
positives = df[is_gold].copy()
print(f"{len(positives)} gold positives")
print(positives["dev_stat"].value_counts())   # the trust gradient for D6 weighting

509 gold positives
dev_stat
Past Producer    300
Prospect         178
Occurrence        27
Unknown            3
Producer           1
Name: count, dtype: int64


In [4]:
print(f"Featurizing {len(positives)} positives...")

rows = []
for i, rec in positives.reset_index(drop=True).iterrows():
    geo = get_geology(rec["latitude"], rec["longitude"])
    rows.append({
        "lat": rec["latitude"],
        "lng": rec["longitude"],
        "site_name": rec["site_name"],
        "dev_stat": rec["dev_stat"],      # D6: trust kept as column -> training weight
        "label": 1,
        **geo,
    })
    if i % 25 == 0:
        print(f"{i}/{len(positives)}")
    time.sleep(0.4)

pos_table = pd.DataFrame(rows)
pos_table.to_csv(PROCESSED / "positives_gold.csv", index=False)
pos_table["status"].value_counts()

Featurizing 509 positives...
0/509
25/509
50/509
75/509
100/509
125/509
150/509
175/509
200/509
225/509
250/509
275/509
300/509
325/509
350/509
375/509
400/509
425/509
450/509
475/509
500/509


status
OK    509
Name: count, dtype: int64

In [5]:
# NEXT (Session B): pseudo-absences — GA polygon (census shapefile),
# rejection sampling, YOUR DECISION #7 (how many? exclusion buffer?),
# featurize with label=0. Cache is warm for any point near a positive.